# N-Gram Spell Correction

**Goal:** Suggest the correct spelling of a misspelled word by comparing character n-grams.

---

## Theory

An **n-gram** is a sequence of `n` consecutive characters from a word.

```
Word: 'cat'  with n=2  →  bigrams: { #c, ca, at, t# }
      (# marks word boundaries — important for matching word edges)
```

**Jaccard Similarity** measures overlap between two sets:

```
score = |A ∩ B| / |A ∪ B|

Range: 0.0 (nothing in common) → 1.0 (identical)
```

The dictionary word with the **highest Jaccard score** against the misspelled word is the best suggestion.

### Example

| Word | Bigrams | Shared with "tecnology" | Score |
|------|---------|------------------------|-------|
| technology | {#t, te, ec, ch, ...} | high overlap | ~0.75 |
| computer | {#c, co, om, ...} | low overlap | ~0.10 |

In [1]:
def generate_ngrams(word: str, n: int = 2) -> set:
    """
    Generate character n-grams for a word with boundary markers.

    Args:
        word: Input word
        n:    Size of each gram (default: bigrams, n=2)

    Returns:
        Set of n-gram strings

    Example:
        generate_ngrams('cat', 2) → {'#c', 'ca', 'at', 't#'}
    """
    padded = "#" + word.lower() + "#"
    return {padded[i:i + n] for i in range(len(padded) - n + 1)}


def jaccard_similarity(set_a: set, set_b: set) -> float:
    """Jaccard similarity between two sets: |A∩B| / |A∪B|"""
    if not set_a and not set_b:
        return 1.0
    return len(set_a & set_b) / len(set_a | set_b)


def correct_spelling(misspelled: str, dictionary: list, n: int = 2) -> tuple:
    """
    Find the closest word in dictionary by n-gram Jaccard similarity.

    Returns:
        (best_match: str, score: float)
    """
    query_grams = generate_ngrams(misspelled, n)

    ranked = sorted(
        [(jaccard_similarity(query_grams, generate_ngrams(w, n)), w) for w in dictionary],
        reverse=True
    )

    return ranked[0][1], ranked[0][0]   # (best word, score)


In [2]:
# ── Quick inspection of n-gram generation ──────────────────
word = "technology"
print(f"Bigrams of '{word}':", sorted(generate_ngrams(word, 2)))
print(f"Trigrams of '{word}':", sorted(generate_ngrams(word, 3)))


Bigrams of 'technology': ['#t', 'ch', 'ec', 'gy', 'hn', 'lo', 'no', 'og', 'ol', 'te', 'y#']
Trigrams of 'technology': ['#te', 'chn', 'ech', 'gy#', 'hno', 'log', 'nol', 'ogy', 'olo', 'tec']


In [3]:
# ── Spell correction tests ──────────────────────────────────
dictionary = [
    "computer", "science", "engineering", "information",
    "technology", "spelling", "correction", "program"
]

test_words = ["tecnology", "sceince", "progrm", "enginring", "infomation"]

print(f"{'Misspelled':<15} {'Suggestion':<15} {'Score':>6}")
print("-" * 40)

for word in test_words:
    suggestion, score = correct_spelling(word, dictionary)
    print(f"{word:<15} {suggestion:<15} {score:>6.2f}")


Misspelled      Suggestion       Score
----------------------------------------
tecnology       technology        0.75
sceince         science           0.50
progrm          program           0.67
enginring       engineering       0.64
infomation      information       0.77


## What to Try Next
- Switch `n=2` to `n=3` — does accuracy improve on longer words?
- Try **Levenshtein distance** (`pip install python-Levenshtein`) — compare results
- Build a simple autocorrect that reads a sentence, finds each unknown word, and suggests fixes